# 01 — Data Ingestion

Downloads NHANES survey files directly from the CDC.

NHANES runs every two years, and each cycle's files live at a predictable URL
that encodes the starting year and a letter suffix (J = 2017-18, I = 2015-16,
and so on). Rather than clicking through 22 download pages, this notebook
builds those URLs programmatically so the whole dataset can be rebuilt with
one command.

Two files per cycle:
- **DEMO** — demographics: age, gender, income, survey weights
- **DR1TOT** — dietary totals: caffeine intake in mg, from a 24-hour recall interview 

In [1]:
import requests
import pandas as pd
from pathlib import Path

# Each NHANES cycle maps to (year folder in the URL, file suffix).
# The suffix letters run alphabetically from the 2001-02 cycle onward.
# 1999-2000 predates the lettering, so its files carry no suffix at all.
CYCLES = {
    "1999-2000": ("1999", ""),
    "2001-2002": ("2001", "B"),
    "2003-2004": ("2003", "C"),
    "2005-2006": ("2005", "D"),
    "2007-2008": ("2007", "E"),
    "2009-2010": ("2009", "F"),
    "2011-2012": ("2011", "G"),
    "2013-2014": ("2013", "H"),
    "2015-2016": ("2015", "I"),
    "2017-2018": ("2017", "J"),
    "2021-2023": ("2021", "L"),
}

# URL template. The two braces get filled in per cycle.
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{filename}.xpt"

# Raw downloads land here. The notebook lives in notebooks/, so step up one level.
RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

print(f"{len(CYCLES)} cycles configured")
print(f"Raw data folder: {RAW.resolve()}")

11 cycles configured
Raw data folder: /Users/hargunkaurkohli/Desktop/caffeine-forecast/data/raw


## Download helper

One function, one file. It returns a status dictionary rather than printing,
so the caller can collect results into a log table.

Two design choices worth noting:
- **Idempotent**: if the file is already on disk it skips the download, so
  re-running the notebook is cheap and safe.
- **Fails soft**: a missing or renamed file returns an error status instead of
  raising, so one bad URL cannot kill a 22-file run. The failures show up in
  the log for us to investigate.

In [2]:
def download_file(filename: str, year: str, overwrite: bool = False) -> dict:
    """Download one NHANES .xpt file into data/raw/ and report what happened."""
    url = BASE_URL.format(year=year, filename=filename)
    dest = RAW / f"{filename}.xpt"

    # Already have it? Don't re-download.
    if dest.exists() and not overwrite:
        size = dest.stat().st_size / 1_000_000
        return {"file": filename, "status": "skipped", "size_mb": round(size, 2), "url": url}

    # Network problems (timeout, DNS, no connection) shouldn't crash the loop.
    try:
        response = requests.get(url, timeout=60)
    except requests.RequestException as e:
        return {"file": filename, "status": f"error: {type(e).__name__}", "size_mb": 0.0, "url": url}

    # A wrong URL returns a 404 page, not a file. Catch that before saving.
    if response.status_code != 200:
        return {"file": filename, "status": f"HTTP {response.status_code}", "size_mb": 0.0, "url": url}

    dest.write_bytes(response.content)
    size = dest.stat().st_size / 1_000_000
    return {"file": filename, "status": "downloaded", "size_mb": round(size, 2), "url": url}

In [3]:
download_file("DEMO_J", "2017")

{'file': 'DEMO_J',
 'status': 'downloaded',
 'size_mb': 3.41,
 'url': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.xpt'}

## Fetch all cycles

Two files per cycle, DEMO and DR1TOT, across every cycle in `CYCLES`.
Results collect into a dataframe so failures are visible at a glance rather
than buried in scroll-back.

Expect some failures. NHANES file naming is not perfectly consistent across
25 years, and the failures tell us exactly where it breaks.

In [10]:
def dietary_stem(year: str) -> str:
    """
    NHANES introduced a second 24-hour recall in 2003 and renamed the totals
    file accordingly. Cycles before that have a single recall named DRXTOT.
    """
    return "DRXTOT" if int(year) < 2003 else "DR1TOT"

In [13]:
# A pipeline that doesn't check itself isn't a pipeline.
failed = log[~log["status"].isin(["downloaded", "skipped"])]

print(f"Files on disk:  {len(list(RAW.glob('*.xpt')))}")
print(f"Total size:     {log['size_mb'].sum():.1f} MB")
print(f"Failures:       {len(failed)}")

if len(failed):
    display(failed[["cycle", "file", "status"]])

log.to_csv(RAW / "_download_log.csv", index=False)
print(f"\nLog written to {RAW / '_download_log.csv'}")

Files on disk:  22
Total size:     181.9 MB
Failures:       0

Log written to ../data/raw/_download_log.csv


In [14]:
log.to_csv("../data/download_log.csv", index=False)
print(f"\nLog written to ../data/download_log.csv")


Log written to ../data/download_log.csv
